# 🎯 Week 3 Lab: OOP Motor Controllers
## Sensory Feedback Models: EP, OFC, and VITE

**Course:** Machine Learning for Neuroscience
**Duration:** ~90 minutes

---

## Learning Objectives

By the end of this lab, you will be able to:
1. **Define an abstract base class** (ABC) with abstract methods
2. **Implement inheritance** to create three motor controller subclasses
3. **Use polymorphism** to compare models through a shared interface
4. **Encapsulate** model-specific state inside each controller class
5. **Explain** how the EP, OFC, and VITE models incorporate sensory feedback
6. **Compare** model predictions for velocity profiles and perturbation recovery

## OOP Concepts Covered

| Concept | Where You’ll Use It |
|---|---|
| Abstract Base Class | `MotorController(ABC)` with `@abstractmethod` |
| Inheritance | `EPController(MotorController)`, etc. |
| Polymorphism | `for m in models: m.simulate()` |
| Encapsulation | λ trajectory, time-varying gains, PPC hidden inside classes |
| Method Override | Each subclass implements `compute_torque()` |
| `super().__init__()` | Calling parent constructor |

## Bloom’s Taxonomy Roadmap

| Level | Section | What You’ll Do |
|-------|---------|----------------|
| 🟢 **Remember** | Part 1 | Define ABC with `simulate()` and `compute_torque()` |
| 🟡 **Understand** | Part 2 | Implement EP model (λ-shift + spring) |
| 🟠 **Apply** | Parts 3–4 | Implement OFC and VITE models |
| 🟦 **Analyze** | Parts 5–6 | Compare models via polymorphism + perturbation |
| 🟣 **Evaluate** | Part 7 | Parameter sensitivity (VITE GO signal) |
| 🔴 **Create** | Part 8 | Design your own controller subclass |

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'lines.linewidth': 1.5})
np.random.seed(42)
print("Setup complete!")

## Physical Parameters

In [ ]:
# Physical parameters (shared across all models)
dt = 0.001; T = 1.5; time_vec = np.arange(0, T, dt)
L = 0.35       # forearm length (m)
m_arm = 1.5    # forearm mass (kg)
B_damp = 0.5   # joint damping (N*m*s/rad)
I_inertia = (1/3) * m_arm * L**2  # moment of inertia

theta_start = np.deg2rad(45)  # initial angle
theta_target = np.deg2rad(90)  # target angle
print(f"I = {I_inertia:.4f} kg*m^2, Target: 45deg -> 90deg")

---
## 🟢 Part 1: Abstract Base Class

### Exercise 1.1: Define MotorController
Complete the abstract base class with `compute_torque()` as an abstract method and Euler integration in `simulate()`.

In [ ]:
# Solution 1.1: Define the abstract base class
class MotorController(ABC):
    """Abstract base class for motor control models."""
    
    def __init__(self, name, dt=0.001, T=1.5, I=I_inertia, B=B_damp,
                 theta_start=theta_start, theta_target=theta_target):
        self.name = name
        self.dt = dt
        self.T = T
        self.I = I
        self.B = B
        self.theta_start = theta_start
        self.theta_target = theta_target
        self.time = np.arange(0, T, dt)
        self.n_steps = len(self.time)
        # State arrays (filled by simulate)
        self.theta = None
        self.omega = None
        self.torque_history = None
    
    ### YOUR CODE HERE ###
    # Define compute_torque as an @abstractmethod
    # It should take self, theta, omega, step and return torque
    pass
    
    def simulate(self, perturb_time=None, perturb_mag=0.0):
        """Euler integration with optional perturbation."""
        self.theta = np.zeros(self.n_steps)
        self.omega = np.zeros(self.n_steps)
        self.torque_history = np.zeros(self.n_steps)
        self.theta[0] = self.theta_start
        perturbed = False
        
        for i in range(self.n_steps - 1):
            if perturb_time and not perturbed and self.time[i] >= perturb_time:
                self.theta[i] += perturb_mag
                perturbed = True
            
            ### YOUR CODE HERE ###
            # 1. Call self.compute_torque(theta[i], omega[i], i) to get tau
            # 2. Compute angular acceleration: alpha = (tau - B*omega) / I
            # 3. Euler update omega[i+1] and theta[i+1]
            pass
        return self.theta, self.omega
    
    def plot_results(self, ax=None, color='blue', label=None):
        """Plot position and velocity on given or new axes."""
        if self.theta is None:
            raise RuntimeError("Call simulate() first!")
        lbl = label or self.name
        if ax is None:
            fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
        else:
            axes = ax if hasattr(ax, '__len__') else [ax]
        
        if len(axes) >= 2:
            axes[0].plot(self.time*1000, np.rad2deg(self.theta), color=color, lw=2, label=lbl)
            axes[0].axhline(90, color='gray', ls='--', alpha=0.4)
            axes[0].set_ylabel('Angle (deg)'); axes[0].legend(fontsize=9)
            axes[1].plot(self.time*1000, np.rad2deg(self.omega), color=color, lw=2, label=lbl)
            axes[1].axhline(0, color='gray', alpha=0.3)
            axes[1].set_xlabel('Time (ms)'); axes[1].set_ylabel('Velocity (deg/s)')
            axes[1].legend(fontsize=9)
        return axes
    
    def get_metrics(self):
        """Compute movement quality metrics."""
        th_deg = np.rad2deg(self.theta)
        om_deg = np.rad2deg(self.omega)
        error = abs(th_deg[-1] - np.rad2deg(self.theta_target))
        pk_v = np.max(np.abs(om_deg))
        moving = np.abs(om_deg) > 5.0
        if np.any(moving):
            on = np.argmax(moving)
            off = len(moving) - 1 - np.argmax(moving[::-1])
            mt = self.time[off] - self.time[on]
        else:
            mt = 0.0
        return {'endpoint_error': error, 'peak_velocity': pk_v, 'movement_time_ms': mt*1000}

# Test: cannot instantiate ABC
try:
    m = MotorController("test")
    print("ERROR: should not instantiate ABC!")
except TypeError as e:
    print(f"\u2705 ABC works: {e}")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

- Use `@abstractmethod` decorator before `def compute_torque(self, theta, omega, step):`
- In simulate: `tau = self.compute_torque(self.theta[i], self.omega[i], i)`
- Euler: `alpha = (tau - self.B*self.omega[i])/self.I`; `self.omega[i+1] = self.omega[i] + alpha*self.dt`; `self.theta[i+1] = self.theta[i] + self.omega[i]*self.dt`
</details>

---
## 🟡 Part 2: Equilibrium Point Controller

### Exercise 2.1: Implement EPController
Complete `compute_torque()` using the spring-like restoring force: τ = K · (λ − θ).

In [ ]:
# Solution 2.1: Equilibrium Point Controller
class EPController(MotorController):
    """Feldman's Equilibrium Point (lambda) model.
    
    The CNS shifts the stretch reflex threshold lambda(t) from start to target.
    Muscle-reflex uses DELAYED sensory feedback (~30ms proprioceptive loop):
    tau = K * (lambda - theta_delayed) - B_reflex * omega_delayed
    """
    
    def __init__(self, K=30.0, B_reflex=1.5, shift_duration=0.3,
                 sensory_delay=0.03, **kwargs):
        super().__init__(name="EP (\u03bb-model)", **kwargs)
        self.K = K              # muscle-reflex stiffness (N*m/rad)
        self.B_reflex = B_reflex  # velocity-sensitive damping
        self.shift_duration = shift_duration
        self.sensory_delay = sensory_delay
        self.delay_steps = int(sensory_delay / self.dt)
        # Pre-compute lambda trajectory
        self.lambda_traj = self._compute_lambda()
    
    def _compute_lambda(self):
        """Sigmoidal lambda shift from start to target."""
        dur = self.shift_duration
        x = (self.time - dur/2) / (dur/8)
        lam = self.theta_start + (self.theta_target - self.theta_start) / (1 + np.exp(-x))
        return lam
    
    def compute_torque(self, theta, omega, step):
        """Stretch reflex with proprioceptive delay: position + velocity components."""
        ### YOUR CODE HERE ###
        # 1. Get delayed sensory index: sensed_step = max(0, step - self.delay_steps)
        # 2. Read delayed signals: theta_sensed = self.theta[sensed_step]
        #                          omega_sensed = self.omega[sensed_step]
        # 3. Return K * (lambda_traj[step] - theta_sensed) - B_reflex * omega_sensed
        pass

# Test
ep = EPController()
ep.simulate()
met = ep.get_metrics()
print(f"EP Model: error={met['endpoint_error']:.1f}\u00b0, peak_v={met['peak_velocity']:.0f}\u00b0/s, MT={met['movement_time_ms']:.0f}ms")
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(ep.time*1000, np.rad2deg(ep.lambda_traj), 'r--', lw=1.5, label='\u03bb(t)')
axes[0].axvline(30, color='#e74c3c', ls=':', alpha=0.5, lw=1)
axes[0].text(35, np.rad2deg(ep.theta_start)+3, '30 ms\nreflex delay', fontsize=8, color='#e74c3c', style='italic')
ep.plot_results(axes, color='#3498db')
axes[0].set_title('EP Model: \u03bb-Shift + Delayed Stretch Reflex', fontweight='bold')
axes[0].set_xlim(0, 700)
plt.tight_layout(); plt.show()
print("\u2705 Exercise 2.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`sensed_step = max(0, step - self.delay_steps)`
`theta_sensed = self.theta[sensed_step]`
`omega_sensed = self.omega[sensed_step]`
`return self.K * (self.lambda_traj[step] - theta_sensed) - self.B_reflex * omega_sensed`
</details>

---
## 🟠 Part 3: Optimal Feedback Controller

### Exercise 3.1: Implement OFCController
Complete the control law using the state estimate from the Kalman filter.

In [ ]:
# Solution 3.1: Optimal Feedback Controller
class OFCController(MotorController):
    """Simplified Optimal Feedback Controller (LQR-inspired).
    
    Deterministic OFC for point-to-point reaching:
    State vector: x = [theta, omega]
    Control law:  u(t) = Kp(t)*(target - theta) - Kd(t)*omega
    
    Gains follow sigmoid ramps from ~0 to Kp_final/Kd_final, each with
    DIFFERENT timing (a genuine Riccati solution property):
      - Kd (velocity/damping) ramps up EARLY to stabilize the limb
      - Kp (position) ramps up LATER, increasing near the deadline
    Includes 30ms sensorimotor delay before first control output.
    """
    
    def __init__(self, Kp_final=120.0, Kd_final=12.0, T_deadline=0.45,
                 sig_center_kp=0.40, sig_spread_kp=0.10,
                 sig_center_kd=0.20, sig_spread_kd=0.12,
                 max_torque=60.0, sensorimotor_delay=0.03, **kwargs):
        super().__init__(name="OFC (LQR)", **kwargs)
        self.Kp_final = Kp_final
        self.Kd_final = Kd_final
        self.T_deadline = T_deadline
        self.sig_center_kp = sig_center_kp
        self.sig_spread_kp = sig_spread_kp
        self.sig_center_kd = sig_center_kd
        self.sig_spread_kd = sig_spread_kd
        self.max_torque = max_torque
        self.sensorimotor_delay = sensorimotor_delay
    
    def simulate(self, **kwargs):
        """Override to initialize gain tracking."""
        self.gain_Kp = np.zeros(self.n_steps)
        self.gain_Kd = np.zeros(self.n_steps)
        return super().simulate(**kwargs)
    
    def compute_torque(self, theta, omega, step):
        """Time-varying optimal feedback with sensorimotor delay."""
        # Sensorimotor delay: no control output during initial delay
        if self.time[step] < self.sensorimotor_delay:
            return 0.0
        
        # Time-varying gains: SEPARATE sigmoid ramps (Riccati property)
        # Kd ramps early (damping first), Kp ramps later (position correction near deadline)
        progress = min((self.time[step] - self.sensorimotor_delay) / self.T_deadline, 1.0)
        scale_kp = 1.0 / (1.0 + np.exp(-(progress - self.sig_center_kp) / self.sig_spread_kp))
        scale_kd = 1.0 / (1.0 + np.exp(-(progress - self.sig_center_kd) / self.sig_spread_kd))
        
        Kp = self.Kp_final * scale_kp
        Kd = self.Kd_final * scale_kd
        self.gain_Kp[step] = Kp
        self.gain_Kd[step] = Kd
        
        # Control law: tau = Kp*(target - theta) - Kd*omega
        error = self.theta_target - theta
        tau = Kp * error - Kd * omega
        return np.clip(tau, -self.max_torque, self.max_torque)

# Test
ofc = OFCController()
ofc.simulate()
met = ofc.get_metrics()
print(f"OFC Model: error={met['endpoint_error']:.1f}\u00b0, peak_v={met['peak_velocity']:.0f}\u00b0/s, MT={met['movement_time_ms']:.0f}ms")

# 3-panel detail: position, velocity, time-varying gains
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)

# Panel 1: Position
axes[0].plot(ofc.time*1000, np.rad2deg(ofc.theta), color='#2E8B8B', lw=2, label='Actual \u03b8(t)')
axes[0].axhline(90, color='gray', ls='--', alpha=0.4, label='Target')
axes[0].axvline(30, color='#e74c3c', ls=':', alpha=0.5, lw=1)
axes[0].text(38, 50, '30 ms\ndelay', fontsize=8, color='#e74c3c', style='italic')
axes[0].set_ylabel('Angle (\u00b0)'); axes[0].legend(fontsize=8, loc='center right')
axes[0].set_title('OFC Model: Time-Varying Gains + Sensorimotor Delay', fontweight='bold')

# Panel 2: Clean bell-shaped velocity
axes[1].plot(ofc.time*1000, np.rad2deg(ofc.omega), color='#2E8B8B', lw=2)
axes[1].set_ylabel('Velocity (\u00b0/s)'); axes[1].axhline(0, color='gray', alpha=0.3)

# Panel 3: Time-varying gains on dual axes (Kd ramps earlier, Kp later)
ax_kp = axes[2]
ax_kd = ax_kp.twinx()
ln1 = ax_kp.plot(ofc.time*1000, ofc.gain_Kp, color='#e67e22', lw=2.5, label='K$_p$(t)')
ax_kp.set_ylabel('Position gain K$_p$', color='#e67e22', fontsize=10)
ax_kp.tick_params(axis='y', labelcolor='#e67e22')
ln2 = ax_kd.plot(ofc.time*1000, ofc.gain_Kd, color='#e74c3c', lw=2.5, ls='--', label='K$_d$(t)')
ax_kd.set_ylabel('Velocity gain K$_d$', color='#e74c3c', fontsize=10)
ax_kd.tick_params(axis='y', labelcolor='#e74c3c')
ax_kp.set_xlabel('Time (ms)')
lns = ln1 + ln2; labs = [l.get_label() for l in lns]
ax_kp.legend(lns, labs, fontsize=9, loc='center right')
ax_kp.text(350, ofc.gain_Kp.max()*0.4, 'K$_d$ ramps early\n(damping first)\nK$_p$ ramps later\n(position correction)', fontsize=7, color='#1B3A5C', style='italic')

for ax in axes[:2]: ax.set_xlim(0, 700)
ax_kp.set_xlim(0, 700)
plt.tight_layout(); plt.show()
print("\u2705 Exercise 3.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`error = self.theta_target - theta`
`tau = Kp * error - Kd * omega`
`return np.clip(tau, -self.max_torque, self.max_torque)`
</details>

---
## 🟠 Part 4: VITE Controller

### Exercise 4.1: Implement VITEController
Complete the difference vector computation and PPC integration.

In [ ]:
# Solution 4.1: VITE Controller
class VITEController(MotorController):
    """Bullock & Grossberg's VITE (Vector Integration to Endpoint) model.
    
    TPC: Target Position Command (fixed)
    PPC: Present Position Command (updated via integration)
    DV: Difference Vector = TPC - PPC
    dPPC/dt = alpha * GO * DV
    """
    
    def __init__(self, alpha=8.0, GO=1.0, K_spring=30.0, **kwargs):
        super().__init__(name="VITE", **kwargs)
        self.alpha = alpha      # integration rate
        self.GO = GO            # go signal magnitude
        self.K_spring = K_spring  # muscle spring converting PPC to torque
        # Internal state
        self.PPC = None
        self.DV_history = None
    
    def simulate(self, **kwargs):
        """Override to initialize PPC and DV tracking."""
        self.PPC = np.zeros(self.n_steps)
        self.DV_history = np.zeros(self.n_steps)
        self.PPC[0] = self.theta_start
        return super().simulate(**kwargs)
    
    def compute_torque(self, theta, omega, step):
        """VITE: integrate difference vector, generate spring-like torque."""
        ### YOUR CODE HERE ###
        # 1. Compute DV = theta_target - PPC[step]
        # 2. Store in DV_history[step]
        # 3. Update PPC[step+1] = PPC[step] + alpha * GO * DV * dt  (if step < n_steps-1)
        # 4. Return K_spring * (PPC[step] - theta)
        pass

# Test
vite = VITEController()
vite.simulate()
met = vite.get_metrics()
print(f"VITE Model: error={met['endpoint_error']:.1f}\u00b0, peak_v={met['peak_velocity']:.0f}\u00b0/s, MT={met['movement_time_ms']:.0f}ms")
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
axes[0].plot(vite.time*1000, np.rad2deg(vite.PPC), 'g-', lw=2, label='PPC (internal plan)')
axes[0].axhline(90, color='blue', ls='--', lw=1, label='TPC (target)')
axes[0].plot(vite.time*1000, np.rad2deg(vite.theta), 'orange', lw=2, label='Actual \u03b8 (limb)')
axes[0].set_ylabel('Angle (\u00b0)'); axes[0].legend(fontsize=8)
axes[0].set_title('VITE Model: DV = TPC \u2212 PPC (internal planning error)', fontweight='bold')
# Panel 2: DV = TPC-PPC AND the visible gap TPC-theta
axes[1].plot(vite.time*1000, np.rad2deg(vite.DV_history), 'purple', lw=2, label='DV = TPC \u2212 PPC')
visible_gap = vite.theta_target - vite.theta
axes[1].plot(vite.time*1000, np.rad2deg(visible_gap), 'orange', lw=1.5, ls='--', label='TPC \u2212 \u03b8 (visible gap)')
axes[1].set_ylabel('Difference (\u00b0)'); axes[1].axhline(0, color='gray', alpha=0.3)
axes[1].legend(fontsize=8)
axes[1].text(250, 10, 'DV tracks internal\nplanning error,\nnot limb position', fontsize=8, color='purple', style='italic')
axes[2].plot(vite.time*1000, np.rad2deg(vite.omega), 'orange', lw=2)
axes[2].set_ylabel('Velocity (\u00b0/s)'); axes[2].set_xlabel('Time (ms)')
axes[2].axhline(0, color='gray', alpha=0.3)
for ax in axes: ax.set_xlim(0, 700)
plt.tight_layout(); plt.show()
print("\u2705 Exercise 4.1 passed!")

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

`DV = self.theta_target - self.PPC[step]`
`self.DV_history[step] = DV`
`if step < self.n_steps - 1: self.PPC[step+1] = self.PPC[step] + self.alpha*self.GO*DV*self.dt`
`return self.K_spring * (self.PPC[step] - theta)`
</details>

---
## 🟦 Part 5: Polymorphic Comparison

### Exercise 5.1: Compare All Three Models
This code is complete — run it to see polymorphism in action!

In [ ]:
# Solution 5.1: Polymorphic comparison — use the shared interface
models = [EPController(), OFCController(), VITEController()]
colors = ['#3498db', '#2E8B8B', '#9b59b6']

# Simulate all
for m in models:
    m.simulate()

# Compare velocity profiles
fig, ax = plt.subplots(figsize=(8, 4))
for m, c in zip(models, colors):
    ax.plot(m.time*1000, np.rad2deg(m.omega), color=c, lw=2, label=m.name)
ax.axhline(0, color='gray', alpha=0.3)
ax.set_xlim(0, 700); ax.set_xlabel('Time (ms)'); ax.set_ylabel('Velocity (\u00b0/s)')
ax.set_title('Velocity Profile Comparison (Polymorphism!)', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

# Metrics table
print(f"{'Model':<18} {'Error (\u00b0)':>10} {'Peak V (\u00b0/s)':>14} {'MT (ms)':>10}")
print("-"*55)
for m in models:
    met = m.get_metrics()
    print(f"{m.name:<18} {met['endpoint_error']:>10.1f} {met['peak_velocity']:>14.0f} {met['movement_time_ms']:>10.0f}")

print("\n\u2705 Exercise 5.1 passed!")

---
## 🟦 Part 6: Perturbation Analysis

### Exercise 6.1: Perturbation Recovery
This code is complete — study how each model recovers from a +10° push.

In [ ]:
# Solution 6.1: Perturbation comparison
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
perturb_mag = np.deg2rad(10)

annotations = {
    'EP': 'Spring restoring force:\n\u03b8 pushed from \u03bb \u2192 reflex pulls back',
    'OFC': 'State error detected;\noptimal gain corrects',
    'VITE': 'PPC unaffected by push;\nPPC\u2212\u03b8 mismatch restores',
}

for ax, ModelClass, color, name in zip(axes,
    [EPController, OFCController, VITEController],
    ['#3498db', '#2E8B8B', '#9b59b6'],
    ['EP', 'OFC', 'VITE']):
    
    # Normal
    m_norm = ModelClass()
    m_norm.simulate()
    ax.plot(m_norm.time*1000, np.rad2deg(m_norm.theta), color=color, lw=2, label='Normal')
    
    # Perturbed
    m_pert = ModelClass()
    m_pert.simulate(perturb_time=0.2, perturb_mag=perturb_mag)
    ax.plot(m_pert.time*1000, np.rad2deg(m_pert.theta), color='#e74c3c', lw=2, ls='--', label='Perturbed (+10\u00b0)')
    
    ax.axhline(90, color='gray', ls='--', alpha=0.4)
    ax.axvline(200, color='gray', ls=':', alpha=0.5)
    ax.set_xlim(0, 700); ax.set_ylim(40, 110)
    ax.set_xlabel('Time (ms)'); ax.set_ylabel('Angle (\u00b0)')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    # Add mechanism annotation
    ax.text(420, 100, annotations[name], fontsize=7, color=color, style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.8))

fig.suptitle('Perturbation Recovery (+10\u00b0 at 200 ms)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
print("\u2705 Exercise 6.1 passed!")

---
## 🟣 Part 7: Parameter Exploration

### Exercise 7.1: VITE GO Signal Sweep
This code is complete — observe how the GO signal controls speed without changing trajectory shape.

In [ ]:
# Solution 7.1: VITE GO signal parameter sweep
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
go_values = [0.5, 1.0, 1.5, 2.0]
cmap = plt.cm.viridis(np.linspace(0.2, 0.9, len(go_values)))

for go, col in zip(go_values, cmap):
    v = VITEController(GO=go)
    v.simulate()
    axes[0].plot(v.time*1000, np.rad2deg(v.theta), color=col, lw=2, label=f'GO={go}')
    axes[1].plot(v.time*1000, np.rad2deg(v.omega), color=col, lw=2, label=f'GO={go}')

axes[0].axhline(90, color='gray', ls='--', alpha=0.4)
axes[0].set_xlim(0, 700); axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Angle (\u00b0)')
axes[0].set_title('Position vs GO Signal', fontweight='bold'); axes[0].legend(fontsize=9)
axes[1].axhline(0, color='gray', alpha=0.3)
axes[1].set_xlim(0, 700); axes[1].set_xlabel('Time (ms)'); axes[1].set_ylabel('Velocity (\u00b0/s)')
axes[1].set_title('Velocity vs GO Signal', fontweight='bold'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()
print("\u2705 Exercise 7.1 passed!")

### Exercise 7.2: Interpretation Questions

1. Which model produces the most symmetric velocity profile? Why?
2. Which model recovers fastest from perturbation? What’s the mechanism?
3. Why does increasing the VITE GO signal increase speed without changing the endpoint?
4. How does the EP model achieve equifinality (same endpoint despite perturbation)?

**Your Answers:**

1. _[Your answer]_
2. _[Your answer]_
3. _[Your answer]_
4. _[Your answer]_

---
## 🔴 Part 8: Create Your Own Model

### Exercise 8.1: Extend the Hierarchy
Create a new controller subclass. Suggestion: a simple PD controller.

In [ ]:
# Exercise 8.1: Create your own motor controller subclass!
# Ideas: PD controller, Minimum Jerk, or your own feedback law.
# Requirements:
#   - Inherit from MotorController
#   - Override compute_torque()
#   - Call super().__init__() with a name
class MyController(MotorController):
    ### YOUR CODE HERE ###
    pass

# After implementing, test with:
# all_models = [EPController(), OFCController(), VITEController(), MyController()]
# for m in all_models: m.simulate()
# Compare!

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

```python
class PDController(MotorController):
    def __init__(self, Kp=20.0, Kd=5.0, **kwargs):
        super().__init__(name='PD Controller', **kwargs)
        self.Kp = Kp; self.Kd = Kd
    def compute_torque(self, theta, omega, step):
        return self.Kp * (self.theta_target - theta) - self.Kd * omega
```
</details>

---
## 🎯 Lab Summary

| OOP Concept | How You Used It |
|---|---|
| ABC + `@abstractmethod` | `MotorController` with `compute_torque()` |
| Inheritance | `EP`, `OFC`, `VITE` all extend `MotorController` |
| Polymorphism | `for m in models: m.simulate(); m.plot_results()` |
| Encapsulation | λ trajectory, time-varying gains, PPC are private to each class |
| Method Override | Each subclass implements its own `compute_torque()` |
| `super().__init__()` | All subclasses call parent init |

### Key Neuroscience Takeaways

| Model | Feedback Mechanism | Key Prediction |
|---|---|---|
| EP (λ) | Stretch reflex spring | Equifinality: same endpoint despite perturbation |
| OFC | Kalman filter + optimal gains | Minimum intervention: ignore task-irrelevant variability |
| VITE | DV integration + GO gating | Speed control independent of trajectory shape |